# Yeşil/Beton Segmentasyonu — YOLOv8-seg Eğitimi

Bu notebook, drone fotoğraflarında piksel bazlı arazi kaplama segmentasyonu yapan bir **YOLOv8-seg** modeli eğitir: her pikseli **yeşil** (bitki örtüsü) veya **beton** (sert/geçirimsiz zemin) olarak sınıflandırır. Sonuç, `green-concrete-service`'in `/segment` endpoint'inde yüzde hesabı için kullanılır.

**Veri seti konumu:** `Benimle Paylaşılanlar / Drone-proje / dataset / goruntuler_yesil-beton` — içinde `data.yaml`, `train/`, `valid/`, `test/`, `yeniGoruntuler/` bulunuyor (Roboflow export — `README.roboflow.txt` bunu doğruluyor).

**⚠️ Önemli — 'Benimle Paylaşılanlar' klasörleri Colab'da otomatik görünmez.** `dataset` klasörüne zaten kendi Drive'ınıza kısayol eklediyseniz (`Drive'ım` kökünde görünüyorsa) aşağıdaki §2 hücresi `data.yaml`'ı otomatik bulur; eklemediyseniz Drive arayüzünde `dataset` klasörüne sağ tıklayıp **'Kısayol ekle' / 'Add shortcut to Drive'** yapın.

**⚠️ Önemli — bbox mu, polygon mu?** YOLO-seg eğitimi için etiket dosyalarının (`labels/*.txt`) her satırında **poligon köşe noktaları** (4'ten fazla sayı çifti) olması gerekir; sadece 4 sayılık (bounding-box) satırlar varsa bu, **detection** etiketidir ve segmentasyon eğitimi anlamlı sonuç vermez. Bu veri seti Roboflow'dan export edildiği için format "Object Detection" olarak da alınmış olabilir — aşağıdaki §4 hücresi bunu otomatik kontrol eder. Bbox çıkarsa, Roboflow'da aynı projeyi **'Instance Segmentation'** formatında yeniden export almanız gerekir.

**Çıktı:** eğitilmiş `best.pt` — bunu indirip `green-concrete-service/model/best.pt` konumuna koyun ve servisi yeniden başlatın.

## 1) Kurulum + Drive bağlama

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q ultralytics

## 2) Veri seti yolunu bul

`DATASET_YAML`'ı boş bırakırsanız, Drive'ınızda `goruntuler_yesil-beton/data.yaml`'ı otomatik arar. Tam yolu zaten biliyorsanız aşağıya yazıp elle de belirtebilirsiniz.

In [ ]:
DATASET_YAML = ""  # boş = otomatik ara; biliyorsanız tam yolu buraya yazın, ör. "/content/drive/MyDrive/dataset/goruntuler_yesil-beton/data.yaml"

import glob
import os

if not DATASET_YAML:
    search_patterns = [
        "/content/drive/MyDrive/dataset/goruntuler_yesil-beton/data.yaml",  # dataset klasörü MyDrive kökünde kısayol ise
        "/content/drive/MyDrive/**/goruntuler_yesil-beton/data.yaml",       # kısayol başka bir yere eklendiyse
        "/content/drive/**/data.yaml",                                       # son çare: tüm Drive'da ara
    ]
    found = []
    for pattern in search_patterns:
        found = glob.glob(pattern, recursive=True)
        if found:
            break
    if not found:
        raise FileNotFoundError(
            "data.yaml Drive'ınızda bulunamadı.\n"
            "'Benimle Paylaşılanlar > Drone-proje > dataset' klasörüne Drive arayüzünde "
            "sağ tıklayıp 'Kısayol ekle' / 'Add shortcut to Drive' ile kendi Drive'ınıza "
            "ekleyin, sonra bu hücreyi tekrar çalıştırın."
        )
    DATASET_YAML = found[0]

assert os.path.exists(DATASET_YAML), f"data.yaml bulunamadı: {DATASET_YAML}"
print(f"✅ Kullanılacak data.yaml: {DATASET_YAML}")

## 3) Config — tek yerden değiştirin

Sınıf isimleriniz `yesil`/`beton` değilse sorun değil — `green-concrete-service/class_map.py` -> `RAW_TO_LANDCOVER` sözlüğüne kendi isimlerinizi ekleyin (ör. `vegetation`, `pavement`).

In [ ]:
EPOCHS = 100
IMGSZ = 640
BASE_MODEL = "yolov8s-seg.pt"  # hız/doğruluk dengesi: nano (yolov8n-seg.pt) en hızlı ama daha az
                               # doğru; s en iyi denge (çıkarım sunucu tarafında, uçuş sonrası toplu
                               # çalışıyor — gerçek zamanlı değil, bu yüzden doğruluğu önceliklendirdik);
                               # daha da yüksek doğruluk için: yolov8m-seg.pt (daha yavaş eğitim + çıkarım)
RUN_NAME = "green_concrete_segmentation"

# Eğitim biter bitmez best.pt bu klasöre de kopyalanır (Colab oturumu kapanınca kaybolmasın diye)
DRIVE_OUTPUT_DIR = os.path.join(os.path.dirname(DATASET_YAML), "model_ciktilari")

## 4) Veri seti sağlık kontrolü — polygon mu, bbox mu?

`data.yaml`'daki `train` klasöründen rastgele birkaç etiket dosyası okunur; her satırın `class_id x1 y1 x2 y2 ...` biçiminde olması ve **4'ten fazla** koordinat içermesi beklenir (polygon). Tam olarak 4 koordinat (bbox) görülürse aşağıda uyarı basılır.

In [ ]:
import yaml

with open(DATASET_YAML, "r", encoding="utf-8") as f:
    data_cfg = yaml.safe_load(f)

print("Sınıflar (ham isimler):", data_cfg.get("names"))

yaml_dir = os.path.dirname(DATASET_YAML)
train_ref = data_cfg.get("train", "")
train_images_dir = train_ref if os.path.isabs(train_ref) else os.path.join(yaml_dir, train_ref)
train_labels_dir = train_images_dir.replace("images", "labels")

label_files = glob.glob(os.path.join(train_labels_dir, "*.txt"))[:20]
assert label_files, f"Etiket dosyası bulunamadı: {train_labels_dir} — data.yaml'daki 'train' yolunu kontrol edin"

bbox_like, polygon_like = 0, 0
for lf in label_files:
    with open(lf, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 2:
                continue
            num_coords = len(parts) - 1  # ilk sayı class_id
            if num_coords <= 4:
                bbox_like += 1
            else:
                polygon_like += 1

print(f"\nÖrneklenen {len(label_files)} dosyada: {polygon_like} polygon-benzeri satır, {bbox_like} bbox-benzeri satır")
if bbox_like > 0 and polygon_like == 0:
    print("\n⚠️  UYARI: Bu veri seti bounding-box formatında görünüyor, polygon değil.")
    print("    YOLO-seg eğitimi (aşağıdaki §5) çalışabilir ama piksel maskeleri anlamsız/kaba olur.")
    print("    Roboflow'da aynı projeyi 'Instance Segmentation' (polygon) formatında yeniden export alın.")
else:
    print("✅ Veri seti polygon formatında görünüyor, segmentasyon eğitimine uygun.")

## 5) Eğitim

In [ ]:
from ultralytics import YOLO

model = YOLO(BASE_MODEL)

results = model.train(
    data=DATASET_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    name=RUN_NAME,
    patience=20,
)

## 6) Doğrulama (validation seti üzerinde metrikler)

In [ ]:
metrics = model.val()
print("Mask mAP50-95:", metrics.seg.map)
print("Mask mAP50:", metrics.seg.map50)
print("Kategori bazlı mask mAP50:", dict(zip(data_cfg.get("names", []), metrics.seg.maps)))

## 7) Örnek bir görüntüde yüzde hesabını doğrula

Servisin (`app.py`) yapacağı hesabın aynısını burada tek bir örnek üzerinde çalıştırıp mantık kontrolü yapar — servis entegrasyonuna geçmeden önce sonuçların akla yatkın olduğunu görmek için.

In [ ]:
sample_images = glob.glob(os.path.join(train_images_dir, "*.jpg")) + glob.glob(os.path.join(train_images_dir, "*.png"))
assert sample_images, "Örnek görüntü bulunamadı"

sample = sample_images[0]
res = model.predict(sample, conf=0.35, retina_masks=True, verbose=False)[0]

width, height = res.orig_shape[1], res.orig_shape[0]
total_px = width * height
print(f"Örnek: {sample} ({width}x{height})")

if res.masks is not None:
    for i, mask in enumerate(res.masks.data):
        cls_name = res.names[int(res.boxes.cls[i])]
        px = int((mask.cpu().numpy() > 0.5).sum())
        print(f"  instance {i}: sınıf={cls_name}, piksel={px}, yüzde=%{round(px/total_px*100, 2)}")
else:
    print("  Bu örnekte hiçbir maske bulunamadı (eşik değeri çok mu yüksek?)")

## 8) `best.pt`'yi Drive'a kopyala

In [ ]:
import shutil

best_pt_path = os.path.join("runs", "segment", RUN_NAME, "weights", "best.pt")
assert os.path.exists(best_pt_path), f"best.pt bulunamadı: {best_pt_path}"

os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
dest_path = os.path.join(DRIVE_OUTPUT_DIR, "green_concrete_best.pt")
shutil.copy(best_pt_path, dest_path)

print(f"✅ Kopyalandı: {dest_path}")

## 9) Sıradaki adım

1. Yukarıdaki dosyayı (`green_concrete_best.pt`) Drive'ınızdan bilgisayarınıza indirin.
2. `best.pt` olarak yeniden adlandırıp `green-concrete-service/model/best.pt` konumuna koyun.
3. Servisi (yeniden) başlatın: `uvicorn app:app --host 127.0.0.1 --port 8001`
4. `GET /health` çağırıp `"ready": true` döndüğünü doğrulayın.